In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier  # La nuova star
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# SETUP DATI
# Caricamento file Median
df1 = pd.read_csv('android_session_1_1_median.csv')
df2 = pd.read_csv('android_session_2_2_median.csv')
df_full = pd.concat([df1, df2], ignore_index=True).fillna(110)

# Feature e Target
beacon_cols = [c for c in df_full.columns if '-' in c and c not in ['room', 'artwork']]
X = df_full[beacon_cols]
y_artwork = df_full['artwork']

# XGBoost richiede che le classi (target) inizino da 0 e siano numeri consecutivi
le = LabelEncoder()
y_encoded = le.fit_transform(y_artwork)

print(f"Classi codificate: {len(le.classes_)} opere d'arte uniche.")

# Definizione modelli
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# XGBoost
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1
)

# Confronto diretto (10-fold cv)
print("\nAvvio Cross-Validation RF vs XGBoost...")
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores_rf = cross_val_score(rf, X, y_encoded, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Random Forest finito. Media: {scores_rf.mean():.4f}")

scores_xgb = cross_val_score(xgb, X, y_encoded, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"XGBoost finito. Media: {scores_xgb.mean():.4f}")

# Output e grafico
print("\n=== RISULTATI FINALI ===")
print(f"Random Forest: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})")
print(f"XGBoost:       {scores_xgb.mean():.4f} (+/- {scores_xgb.std():.4f})")

results_df = pd.DataFrame({
    'Fold': range(1,11),
    'RandomForest': scores_rf,
    'XGBoost': scores_xgb
})
results_df.to_csv('Confronto_RF_XGBoost.csv', index=False)

# Grafico
plt.figure(figsize=(8, 5))
data_plot = pd.melt(results_df, id_vars=['Fold'], var_name='Algorithm', value_name='Accuracy')
sns.boxplot(x='Algorithm', y='Accuracy', data=data_plot, palette="viridis")
plt.title("Scontro tra Titani: Random Forest vs XGBoost")
plt.grid(True, alpha=0.3)
plt.savefig('RF_vs_XGBoost.png', dpi=300)
plt.show()